# 04 Sampling and Evaluation

Use this notebook to compare baseline training against resampling strategies and threshold adjustments.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fraud_detection.data import load_train_data, split_features_target
from src.fraud_detection.features import build_preprocessor, drop_high_missing_columns
from src.fraud_detection.metrics import compute_classification_metrics

In [ ]:
data = load_train_data(sample_size=30000)
X, y = split_features_target(data)
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, dropped_columns = drop_high_missing_columns(X_train, threshold=0.95)
X_valid = X_valid.drop(columns=dropped_columns, errors="ignore")
preprocessor, _, _ = build_preprocessor(X_train)

In [ ]:
baseline = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                solver="liblinear",
                random_state=42,
            ),
        ),
    ]
)

undersampled_model = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("sampler", RandomUnderSampler(random_state=42)),
        (
            "model",
            LogisticRegression(max_iter=1000, solver="liblinear", random_state=42),
        ),
    ]
)

results = {}
for name, model in {"baseline": baseline, "undersampled": undersampled_model}.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_valid)[:, 1]
    results[name] = compute_classification_metrics(y_valid, scores)

pd.DataFrame(results).T.sort_values("average_precision", ascending=False)

## Follow-Up

- Add SMOTE or SMOTENC once the feature pipeline is adapted for it.
- Add undersampling and hybrid sampling methods.
- Plot precision-recall curves for each strategy.
- Test threshold sweeps instead of using only `0.5`.